# Tune the bootstrap-PF gains with Optuna (full 13-parameter search)

The no-network baseline (`use_net=False`) replaces the 13 controller outputs with
fixed gains. Here we tune **all 13**: one process-noise scale **per state dim** (6),
one prior-regularization weight **per state dim** (6), and a single likelihood
temperature (1). Objective = leave-one-unit-out dev tail-NLL (same metric as
training). Trials are CPU-bound and parallelize across cores via `n_jobs`.

In [1]:
import json
import os

import numpy as np
import optuna
import pandas as pd
import torch

from experiment_config import SEED, DegModel, dataset_paths, pfnet_paths
from src.helpers.seed import set_global_seed
from src.models.particle_filter.core import ParticleFilter
from src.training.pfnet_hparams import PFNET_ARGS


## Configuration

In [2]:
DATA_NAME = "DS06"
EVAL_REPS = 3
N_TRIALS = 600

perform_name = "SmLPC"


In [3]:
ARGS_ID = 0

# PF + eval settings come from the shared config so tuning matches the pipeline
_args = PFNET_ARGS[ARGS_ID]
N_PARTICLES = int(_args["PARTICLE_FILTER"]["N_PARTICLES"])
MAX_LIFE = int(_args["PARTICLE_FILTER"]["MAX_LIFE"])
LOSS_TAIL_STEPS = int(_args["TRAINING"]["LOSS_TAIL_STEPS"])
SEED_STRIDE = int(_args["EVALUATION"]["SEED_STRIDE"])

STATE_DIM = DegModel.state_dim()  # 6 -> 13 gains (6 noise + 6 prior + 1 lik)


# one torch thread per Optuna worker so threads do not oversubscribe the cores
torch.set_num_threads(1)
N_JOBS = int(os.environ.get("SLURM_CPUS_PER_TASK") or os.cpu_count() or 1)

ESTIMATION_DIR, DEGR_MODEL_DIR = dataset_paths(
    DATA_NAME, fields=["estimation", "degr_model"]
)
# per-metric folder, mirroring the training layout net_arg{ARGS_ID}/{perform_name}
PFNET_ARGS_DIR, _ = pfnet_paths(ARGS_ID, DATA_NAME)
PERFORM_DIR = PFNET_ARGS_DIR / perform_name
PERFORM_DIR.mkdir(parents=True, exist_ok=True)
set_global_seed(SEED)
print(f"n_jobs={N_JOBS} | output: {PERFORM_DIR}")


n_jobs=8 | output: /lustre/diazgonz/deep-performance-rul/experiments/DS06/opcond_q0.001-0.999_corr0.6_range0.6/estimation_thr0.2/gamma/net_arg0/SmLPC


## Load dev data and degradation models (once)

In [4]:
dev_hi = pd.read_csv(ESTIMATION_DIR / "data_dev.csv")
dev_units = sorted(dev_hi["unit"].astype(int).unique().tolist())
...


def build_tensors(df, name):
    out = {}
    for u in dev_units:
        sub = df[df["unit"] == u]
        out[u] = torch.tensor(
            np.stack([sub["cycle"].values, sub[name].values], axis=1),
            dtype=torch.float32,
        )
    return out


# single-metric tuning: load only the target performance metric
dev_tensors = build_tensors(dev_hi, perform_name)
dev_degmodels = {}
for u in dev_units:
    m = DegModel()
    m.load_state_dict(
        torch.load(
            DEGR_MODEL_DIR / "states" / perform_name / f"unit_{u}" / "best_model.pt"
        )
    )
    dev_degmodels[u] = m

print("dev units:", dev_units, "| metric:", perform_name)


KeyError: 'SmLPC'

## Tail-NLL evaluation with tunable gains

In [ ]:
def tail_nll(pf, t_data, s_data):
    step_losses = []
    for k in range(len(t_data)):
        mixture = pf.step(t_obs=t_data[[k]], s_obs=s_data[[k]])
        start = -LOSS_TAIL_STEPS if LOSS_TAIL_STEPS else k
        dist = mixture.distribution(s=s_data[start:])
        step_losses.append(-dist.log_prob(t_data[start:]).mean())
    return float(torch.stack(step_losses).mean().item())


@torch.no_grad()
def eval_unit(base_models, unit_tensor, seeds, cn, cp, cl):
    t_data, s_data = unit_tensor[:, 0], unit_tensor[:, 1]
    reps = []
    for sd in seeds:
        with torch.random.fork_rng(devices=[]):
            torch.manual_seed(sd)
            pf = ParticleFilter(
                base_models=base_models,
                net=None,
                n_particles=N_PARTICLES,
                max_life=MAX_LIFE,
                use_net=False,
                const_noise=cn,
                const_prior=cp,
                const_lik=cl,
            ).eval()
            reps.append(tail_nll(pf, t_data, s_data))
    return float(np.mean(reps))

## Objective: leave-one-unit-out dev tail-NLL for a single metric


In [ ]:
def objective(trial):
    cn = [
        trial.suggest_float(f"noise_{d}", 0.05, 5.0, log=True) for d in range(STATE_DIM)
    ]
    cp = [trial.suggest_float(f"prior_{d}", 0.0, 2.0) for d in range(STATE_DIM)]
    cl = trial.suggest_float("lik", 0.1, 10.0, log=True)
    losses = []
    for u in dev_units:
        base = [dev_degmodels[v] for v in dev_units if v != u]
        seeds = [SEED + u * SEED_STRIDE + r for r in range(EVAL_REPS)]
        losses.append(eval_unit(base, dev_tensors[u], seeds, cn, cp, cl))
    return float(np.mean(losses))


## Run the study (parallel, resumable)

In [ ]:
storage = f"sqlite:///{(PERFORM_DIR / 'optuna_pf_gains.db').as_posix()}"
study = optuna.create_study(
    direction="minimize",
    study_name=f"pf_gains_{perform_name}_{DATA_NAME}",
    storage=storage,
    load_if_exists=True,  # resumable + multi-worker safe
    sampler=optuna.samplers.TPESampler(seed=SEED),
)

out_json = PERFORM_DIR / "optuna_best_gains.json"


def _best_gains(study):
    b = study.best_params
    return {
        "NOISE": [round(b[f"noise_{d}"], 4) for d in range(STATE_DIM)],
        "PRIOR": [round(b[f"prior_{d}"], 4) for d in range(STATE_DIM)],
        "LIK": round(b["lik"], 4),
    }


def save_best_gains(study, trial):
    """Write the running-best gains whenever a trial improves (live monitoring)."""
    if study.best_trial.number != trial.number:
        return
    tmp = out_json.with_name(out_json.name + f".{trial.number}.tmp")
    tmp.write_text(json.dumps(_best_gains(study), indent=2))
    os.replace(tmp, out_json)  # atomic swap: readers never see a partial file


# N_TRIALS is the TOTAL target: run only the trials still missing in the db.
# Re-running with the same N_TRIALS just reports; a larger N_TRIALS tops up.
n_done = len(
    study.get_trials(deepcopy=False, states=(optuna.trial.TrialState.COMPLETE,))
)
n_remaining = max(0, N_TRIALS - n_done)
print(
    f"{n_done} completed trials in db; running {n_remaining} more to reach {N_TRIALS}"
)
if n_remaining:
    study.optimize(
        objective,
        n_trials=n_remaining,
        n_jobs=N_JOBS,
        show_progress_bar=True,
        callbacks=[save_best_gains],
    )

b = study.best_params
print(f"best value ({perform_name} mean tail-NLL):", study.best_value)

# final write (also covers resumed runs with no new best during this session)
best_gains = _best_gains(study)
out_json.write_text(json.dumps(best_gains, indent=2))
print("saved:", out_json)

print(f"\n# {perform_name} tuned gains, paste into PFNET_ARGS:")
print(
    f"    gains_=gains(noise={best_gains['NOISE']}, "
    f"prior={best_gains['PRIOR']}, lik={best_gains['LIK']}),"
)


[I 2026-08-06 08:11:45,783] A new study created in RDB with name: pf_gains_SmLPC_DS05


0 completed trials in db; running 600 more to reach 600


  0%|          | 0/600 [00:00<?, ?it/s]

[I 2026-08-06 08:13:34,859] Trial 4 finished with value: 9.510478099187216 and parameters: {'noise_0': 0.08170074207059443, 'noise_1': 0.26550991429556564, 'noise_2': 1.1457155303278808, 'noise_3': 0.14151648963899685, 'noise_4': 0.28000568808427606, 'noise_5': 0.8788383500812513, 'prior_0': 0.2834243196416071, 'prior_1': 0.8370496475266833, 'prior_2': 1.3570416426311211, 'prior_3': 1.576373423037727, 'prior_4': 1.71298501000239, 'prior_5': 0.7222703335221772, 'lik': 0.6635973543301569}. Best is trial 4 with value: 9.510478099187216.
[I 2026-08-06 08:13:35,996] Trial 5 finished with value: 4.032957275708516 and parameters: {'noise_0': 0.08160541237403421, 'noise_1': 1.606634408339163, 'noise_2': 1.475303036593955, 'noise_3': 0.18671290492335843, 'noise_4': 0.09093548555117482, 'noise_5': 1.0899077647472324, 'prior_0': 1.1369230325425206, 'prior_1': 0.9966451710861206, 'prior_2': 1.6907521869327256, 'prior_3': 1.4264469286111112, 'prior_4': 0.579796170530479, 'prior_5': 0.26552244322555

## Sanity check: tuned vs default gains

In [ ]:
def dev_score(cn, cp, cl):
    return float(
        np.mean(
            [
                eval_unit(
                    [dev_degmodels[v] for v in dev_units if v != u],
                    dev_tensors[u],
                    [SEED + u * SEED_STRIDE + r for r in range(EVAL_REPS)],
                    cn,
                    cp,
                    cl,
                )
                for u in dev_units
            ]
        )
    )


b = study.best_params
noise_vec = [b[f"noise_{d}"] for d in range(STATE_DIM)]
prior_vec = [b[f"prior_{d}"] for d in range(STATE_DIM)]
lik = b["lik"]
print("default:", dev_score([1.0] * STATE_DIM, [0.0] * STATE_DIM, 1.0))
print("tuned  :", dev_score(noise_vec, prior_vec, lik))
print("best params:", b)


default: 4.156421687867906
tuned  : 3.332913557688395
best params: {'noise_0': 0.5492760875695069, 'noise_1': 1.9539412925672277, 'noise_2': 0.9637391762655174, 'noise_3': 0.35165649807091987, 'noise_4': 0.0928742290399275, 'noise_5': 0.11605423940827954, 'prior_0': 0.20508858295325744, 'prior_1': 0.2567705790757442, 'prior_2': 1.5393385686622563, 'prior_3': 0.6678859317185896, 'prior_4': 1.1522134732206295, 'prior_5': 1.6605522082065198, 'lik': 1.0885424439963844}
